# 1. Dataset Ingestion & Schema Inspection

### Theory
Initial ingestion reads raw dataset files, establishes primary baseline shapes, and maps initial schema profiles.

### Business Impact
Establishes the entry point for automated data pipelines.

### Risks
Ingesting corrupted text encodings or wrong delimiter formats without validation corrupts downstream tasks.

### Decision Rules
- Inspect raw column data types and non-null counts immediately upon data loading.

In [1]:
import pandas as pd
import numpy as np

# Load Raw Dataset
df_raw = pd.read_csv("Customer_Data.csv")
print(f"Raw Ingestion Shape: {df_raw.shape}")
display(df_raw.head(3))

Raw Ingestion Shape: (1010, 9)


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No


# 2. Primary Key Deduplication & Data Cleaning

### Theory
Identifies duplicate records based on business primary keys and removes redundant entries while standardizing numeric string fields.

### Business Impact
Prevents duplicate customer records from skewing metrics or wasting compute budget.

### Risks
Deleting non-duplicate business records through flawed grouping logic.

### Decision Rules
- Deduplicate based on unique entity identifiers (e.g., `CustomerID`) before performing train/test splits.

In [2]:
# Deduplicate Primary Keys
df_clean = df_raw.drop_duplicates(subset=["CustomerID"], keep="first").copy()

# Format clean numeric values
df_clean["MonthlyCharges"] = pd.to_numeric(
    df_clean["MonthlyCharges"].astype(str).str.replace("$", ""), errors="coerce"
)
df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors="coerce")

print(f"Shape Post-Deduplication & Cleaning: {df_clean.shape}")

Shape Post-Deduplication & Cleaning: (1000, 9)


# 3. Train-Test Splitting

### Theory
Partitions the cleaned dataset into isolated Training and Testing subsets before applying fitted transformations.

### Business Impact
Maintains strict operational isolation to evaluate model generalization accurately.

### Risks
Performing global imputation or scaling before splitting causes data leakage and overoptimistic validation scores.

### Decision Rules
- Perform train/test splitting **before** fitting any imputers, scalers, or encoders.

In [3]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["CustomerID", "Churn"])
y = np.where(df_clean["Churn"] == "Yes", 1, 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")

X_train Shape: (800, 7) | X_test Shape: (200, 7)


# 4. Pipeline Construction & Feature Transformation

### Theory
Assembles numerical and categorical pipelines using `ColumnTransformer` to perform median imputation, robust scaling, and one-hot encoding in a single pass.

### Business Impact
Provides a production-ready preprocessing pipeline that handles raw inputs reliably.

### Risks
Unhandled unseen categorical labels in test/production data throw errors if `handle_unknown='ignore'` is omitted.

### Decision Rules
- Always specify `handle_unknown='ignore'` on OneHotEncoder inside production pipelines.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

num_cols = ["Age", "TenureYears", "MonthlyCharges"]
cat_cols = ["Gender", "ContractType", "PaymentMethod"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

full_pipeline = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Fit strictly on train, transform both
X_train_trans = full_pipeline.fit_transform(X_train)
X_test_trans = full_pipeline.transform(X_test)

print(f"Final Transformed Train Shape: {X_train_trans.shape}")
print(f"Final Transformed Test Shape: {X_test_trans.shape}")

Final Transformed Train Shape: (800, 17)
Final Transformed Test Shape: (200, 17)


# 5. Synthetic Oversampling (SMOTE)

### Theory
Applies SMOTE to balance target class distribution through synthetic sample generation along feature vectors.

### Business Impact
Improves minority class recall in imbalanced business contexts (e.g., fraud, customer churn).

### Risks
Applying SMOTE to the testing set contaminates evaluation data with synthetic samples, producing invalid metrics.

### Decision Rules
- Apply SMOTE **strictly to the training set** after all fitting and transformations are completed.

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_final, y_train_final = smote.fit_resample(X_train_trans, y_train)

print(f"Pre-SMOTE Training Class Counts: {dict(pd.Series(y_train).value_counts())}")
print(f"Post-SMOTE Training Class Counts: {dict(pd.Series(y_train_final).value_counts())}")
print("Notebook 16 execution completed successfully!")